# Cross-Encoder Reranking [Step 2 - The sentence-transformers API]

> **MLCourse - Agentic AI - Advanced RAG - Reranking**

Notebook 01 argued *why* a second stage is needed. This notebook is the
practical one: how the `sentence-transformers` `CrossEncoder` class actually
behaves, how to read its scores, how to batch it so it is not needlessly slow,
how to pick a model, and how to wrap the whole thing in a reusable class you
can drop into any pipeline in this course.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

# Paragraph-sized chunks: human-readable units, good enough for retrieval demos
# and identical to the chunking used in ../01_hybrid_search.
paragraphs = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]

print("characters :", len(raw_text))
print("paragraphs :", len(paragraphs))
print("example    :", paragraphs[10][:150], "...")

characters : 144696
paragraphs : 237
example    : Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passa ...


### 2. Loading a CrossEncoder

`CrossEncoder` wraps a HuggingFace sequence-classification model whose input is
a *pair* of texts and whose output is a single number. The MS MARCO family is
the standard choice for retrieval reranking: those models were trained on real
Bing queries paired with passages that did or did not satisfy them.

`ms-marco-MiniLM-L-6-v2` is 6 transformer layers, about 22M parameters, roughly
80 MB on disk. It downloads once and is cached in `~/.cache/huggingface`.

In [4]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("model      :", reranker.model.config._name_or_path)
print("max tokens :", reranker.tokenizer.model_max_length)
print("num labels :", reranker.model.config.num_labels, "(a single relevance score)")
print("parameters :", f"{sum(p.numel() for p in reranker.model.parameters()):,}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

model      : cross-encoder/ms-marco-MiniLM-L-6-v2
max tokens : 512
num labels : 1 (a single relevance score)
parameters : 22,713,601


### 3. `predict` takes pairs, and only pairs

The API is deliberately small: you pass a list of `(query, document)` tuples and
get back one score per tuple. There is no index, no `add`, no persistence -
a cross-encoder has nothing to store.

Let us start with a hand-made set of pairs where we already know the right
answer, so the numbers become interpretable.

In [5]:
probe_query = "What did the Queen of Hearts shout when she was angry?"

probe_docs = [
    "'Off with her head!' the Queen shouted at the top of her voice.",   # direct answer
    "The Queen of Hearts was fond of playing croquet with flamingoes.",  # same entity, wrong fact
    "Alice thought the whole pack of cards rose up into the air.",       # same book, unrelated
    "The mitochondrion is the powerhouse of the cell.",                  # entirely unrelated
]

scores = reranker.predict([(probe_query, d) for d in probe_docs])

for doc, score in zip(probe_docs, scores):
    print(f"{score:+8.3f}   {doc}")

  +3.571   'Off with her head!' the Queen shouted at the top of her voice.
  -1.194   The Queen of Hearts was fond of playing croquet with flamingoes.
 -10.273   Alice thought the whole pack of cards rose up into the air.
 -10.940   The mitochondrion is the powerhouse of the cell.


### Reading those numbers

The output is a **raw logit**, not a probability. In practice with the MS MARCO
models you will see roughly:

| score range | meaning |
|---|---|
| `> +5` | the passage directly answers the query |
| `0 to +5` | related and probably useful |
| `-5 to 0` | on topic but does not answer |
| `< -5` | irrelevant |

These bands are rules of thumb, not guarantees - they shift between models and
domains. Two consequences follow, and both matter in production:

1. **Never hard-code an absolute threshold** without checking it on your own
   data. `score > 0.5` is a bug waiting to happen.
2. **Comparisons are only valid within one query.** A score of `+3` for query A
   and `+3` for query B do not mean the same thing. Ranking within a query is
   what the model is trained for.

If you genuinely need a 0-1 number - say, to decide "we have nothing good
enough, do not answer" - squash the logits yourself and calibrate the cut on
labelled examples.

In [6]:
import numpy as np


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.asarray(x, dtype=float)))


print("logit -> pseudo-probability (calibrate before trusting it!)")
for doc, score in zip(probe_docs, scores):
    print(f"  {score:+8.3f} -> {sigmoid(score):.4f}   {doc[:60]}...")

logit -> pseudo-probability (calibrate before trusting it!)
    +3.571 -> 0.9727   'Off with her head!' the Queen shouted at the top of her voi...
    -1.194 -> 0.2326   The Queen of Hearts was fond of playing croquet with flaming...
   -10.273 -> 0.0000   Alice thought the whole pack of cards rose up into the air....
   -10.940 -> 0.0000   The mitochondrion is the powerhouse of the cell....


### 4. Batching: the difference between fast and slow

`predict` accepts a `batch_size`. Because the model runs on tensors, scoring 32
pairs in one forward pass is far cheaper than 32 separate passes. This is the
single easiest performance mistake to make - calling `predict` inside a Python
loop, one pair at a time.

In [7]:
query = "What game does the Queen of Hearts make everyone play?"
sample_docs = paragraphs[:64]
pairs = [(query, d) for d in sample_docs]

t0 = time.time()
one_at_a_time = [reranker.predict([p])[0] for p in pairs]
slow_s = time.time() - t0

t0 = time.time()
batched = reranker.predict(pairs, batch_size=32)
fast_s = time.time() - t0

print(f"one pair per call : {slow_s:.2f}s")
print(f"batch_size=32     : {fast_s:.2f}s")
print(f"speedup           : {slow_s / fast_s:.1f}x")
print(f"identical scores  : {np.allclose(one_at_a_time, batched, atol=1e-4)}")

one pair per call : 0.84s
batch_size=32     : 0.72s
speedup           : 1.2x
identical scores  : True


### 5. Truncation is silent - and it will bite you

The model has a fixed input budget (512 tokens for this one), shared between
the query *and* the document. Anything past that is dropped without warning. A
long chunk whose answer sits in the final paragraph can be scored as if that
paragraph did not exist.

Let us prove it: same answer sentence, buried at increasing depth.

In [8]:
answer_sentence = "The Queen of Hearts organised a game of croquet using live flamingoes as mallets."
filler = ("It was a very fine afternoon and the garden was full of roses. " * 1)

for n_filler in [0, 20, 60, 120]:
    doc = (filler * n_filler) + answer_sentence
    n_tokens = len(reranker.tokenizer.encode(doc))
    score = reranker.predict([(query, doc)])[0]
    truncated = "TRUNCATED" if n_tokens > 512 else "fits"
    print(f"filler x{n_filler:3d} | ~{n_tokens:5d} tokens | {truncated:9s} | score {score:+7.3f}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (861 > 512). Running this sequence through the model will result in indexing errors


filler x  0 | ~   21 tokens | fits      | score  +5.267
filler x 20 | ~  301 tokens | fits      | score  -6.109
filler x 60 | ~  861 tokens | TRUNCATED | score -10.737
filler x120 | ~ 1701 tokens | TRUNCATED | score -10.737


The score collapses once the answer falls outside the window. The practical
rules that follow:

- Keep reranked chunks **under roughly 300-400 words**.
- If your chunks are long, rerank on a *representative window* of the chunk and
  return the full chunk to the LLM.
- Front-load context. [`../13_contextual_retrieval/04_contextual_chunk_headers.ipynb`](../13_contextual_retrieval/04_contextual_chunk_headers.ipynb)
  prepends a short summary header for exactly this reason - it survives
  truncation.

### 6. Model choice

| model | layers | speed | quality | when |
|---|---|---|---|---|
| `ms-marco-TinyBERT-L-2-v2` | 2 | fastest | lowest | very tight latency budgets |
| `ms-marco-MiniLM-L-6-v2` | 6 | fast | good | **the default for this course** |
| `ms-marco-MiniLM-L-12-v2` | 12 | ~2x slower | better | quality-first, offline batch |
| `BAAI/bge-reranker-base` | 12 | ~2x slower | better, multilingual | non-English corpora |
| Cohere Rerank / Voyage rerank (API) | - | network-bound | best | when a hosted API is acceptable |

The L-6 model is the usual sweet spot, and it is the one we measure with in
notebook 04.

### 7. A reusable reranker

Everything so far collapses into a small class. This is the shape you want in a
real codebase: it takes an already-retrieved candidate list and returns a
reordered one, knowing nothing about how the candidates were found.

In [9]:
from dataclasses import dataclass, field


@dataclass
class CrossEncoderReranker:
    """Reorders retrieved candidates by cross-encoder relevance.

    Deliberately knows nothing about the first stage: give it any list of
    (doc_id, text) pairs from BM25, dense search, RRF, or a graph walk.
    """

    model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    batch_size: int = 32
    model: CrossEncoder = field(default=None, repr=False)

    def __post_init__(self):
        if self.model is None:
            self.model = CrossEncoder(self.model_name)

    def rerank(self, query, candidates, top_k=5):
        """candidates: list of (doc_id, text). Returns [(doc_id, text, score)]."""
        if not candidates:
            return []
        scores = self.model.predict(
            [(query, text) for _, text in candidates], batch_size=self.batch_size
        )
        ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
        return [(doc_id, text, float(s)) for (doc_id, text), s in ranked[:top_k]]


rr = CrossEncoderReranker(model=reranker)   # reuse the already-loaded model

candidates = [(i, paragraphs[i]) for i in range(len(paragraphs))]
top = rr.rerank("Who does Alice meet at the mad tea party?", candidates, top_k=4)

for rank, (doc_id, text, score) in enumerate(top, 1):
    print(f"#{rank} doc_{doc_id} score={score:+.2f}")
    print("   ", text[:160], "...")

#1 doc_128 score=-1.56
    “It _is_ the same thing with you,” said the Hatter, and here the conversation dropped, and the party sat silent for a minute, while Alice thought over all she c ...
#2 doc_0 score=-1.89
    CHAPTER I. Down the Rabbit-Hole CHAPTER II. The Pool of Tears CHAPTER III. A Caucus-Race and a Long Tale CHAPTER IV. The Rabbit Sends in a Little Bill CHAPTER V ...
#3 doc_126 score=-2.28
    There was a table set out under a tree in front of the house, and the March Hare and the Hatter were having tea at it: a Dormouse was sitting between them, fast ...
#4 doc_133 score=-2.30
    Alice did not quite know what to say to this: so she helped herself to some tea and bread-and-butter, and then turned to the Dormouse, and repeated her question ...


### 8. Put it in front of the LLM

The reranker is only a means to an end: better context. Here is the smallest
possible complete pipeline - rerank, then generate with Groq.

In [10]:
def rag_with_rerank(query, top_k=3):
    top = rr.rerank(query, candidates, top_k=top_k)
    context = "\n\n".join(f"[doc_{i}] {t}" for i, t, _ in top)
    prompt = (
        "Answer the question using ONLY the context below. Quote the specific "
        "detail you relied on. If the context lacks the answer, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    )
    return top, ask(prompt)


q = "Who does Alice meet at the mad tea party?"
top, answer = rag_with_rerank(q)

print("reranked context:", [f"doc_{i} ({s:+.1f})" for i, _, s in top])
print("\nQ:", q)
print("A:", answer)

reranked context: ['doc_128 (-1.6)', 'doc_0 (-1.9)', 'doc_126 (-2.3)']

Q: Who does Alice meet at the mad tea party?
A: Based on the context provided, Alice meets the **March Hare**, the **Hatter**, and a **Dormouse**.

**Specific detail relied on:**
From [doc_126]: "the March Hare and the Hatter were having tea at it: a Dormouse was sitting between them, fast asleep"


### 9. Key takeaways

- `CrossEncoder.predict` takes `(query, document)` pairs and returns one raw
  logit each. There is nothing to index and nothing to persist.
- Always pass the whole list with a `batch_size`; per-pair calls in a Python
  loop are several times slower for identical results.
- Scores are **not** probabilities and are **not** comparable across queries.
  Rank within a query; calibrate before thresholding.
- Input truncation is silent. Keep reranked text short, or front-load the
  important part.
- Wrap the reranker behind a small interface that accepts candidates from any
  first stage - which is exactly what the next notebook exploits.

Next: [`03_rerank_over_hybrid.ipynb`](03_rerank_over_hybrid.ipynb) feeds it the
RRF output from `../01_hybrid_search`.